# Set Up

## Mount Google Drive

Ignore if not using Google Collab:

In [5]:
from google.colab import drive

# mount google drive
drive.mount('/content/drive')
%cd /content/drive/My Drive
!git clone https://github.com/FranciscoLozCoding/cooling_with_code.git
%cd cooling_with_code
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/My Drive
fatal: destination path 'cooling_with_code' already exists and is not an empty directory.
/content/drive/My Drive/cooling_with_code
Already up to date.


## Import Libraries

In [6]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

#data science
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GridSearchCV
import pickle
import shap

#custom tools
from tools.environment import VALID_SPLIT, RANDOM_STATE
from tools.feature_selection import get_feature_importance, plot_feature_importance_comparison
from tools.distribution import plot_target_var_distribution
from tools.preprocess import load_and_preprocess_data

## Import Datasets

In [7]:
#load in dataset
csv_path = 'data/train/300m_buffer_dataset.csv'

# no split
X, Y, scaler = load_and_preprocess_data(csv_path, split=False)

# split
x_train, x_valid, y_train, y_valid, scaler = load_and_preprocess_data(
    csv_path, split=True,
    test_size=VALID_SPLIT, random_state=RANDOM_STATE)

Loading data from data/train/300m_buffer_dataset.csv
Loading data from data/train/300m_buffer_dataset.csv


# Tuning RandomForestRegressor

This notebook is for tuning our RandomForestRegressor. It was first created as a simple model for us to use as a baseline comparison. Now, we will apply what we learned in notebooks 4-6 (eg; [4](/04_EDA.ipynb), [5](/05_preprocessing.ipynb), [6](06_increasing_buffer_zone.ipynb)) to improve the model. For details on the train/test dataset refer to our past notebooks:
- [01_dataset_generation](/01_dataset_generation.ipynb)
- [02_more_dataset_generation](/02_more_dataset_generation.ipynb)

## Feature Selection

First, we will do feature selection. We now have interaction terms, so the dataset is filled with a lot of features that need to be dropped.

In [8]:
# Ensure x_train is a DataFrame
x_train_df = pd.DataFrame(x_train, columns=X.columns)

# Base Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)

# Recursive Feature Elimination with Cross-Validation
rfe = RFECV(estimator=rf, step=1, cv=5, scoring="r2", n_jobs=-1)

# Fit to training data
rfe.fit(x_train_df, y_train)

# Get selected features
selected_features = x_train_df.loc[:, rfe.support_]

# Get names of selected features
selected_feature_names = x_train_df.columns[rfe.support_]

# Print selected feature names
print("Selected Features:", list(selected_feature_names))

Selected Features: ['NDVI', 'SI', 'NPCRI', 'Coastal_Aerosol', 'Building_Count', 'Total_Building_Area_m2', 'Building_Construction_Year', 'Ground_Elevation', 'Traffic_Volume', 'Building_Wind_X', 'Building_Wind_Y', 'Elevation_Wind_Y', 'BldgHeight_Count', 'TotalBuildingArea_NDVI', 'Traffic_NDVI', 'Traffic_NDBI', 'Traffic_BuildingDensity']


Here we see the features that were selected. These features maximized the cross-validation score. Now let's select them for our training/validation set.

In [10]:
# select features
x_train_selected = pd.DataFrame(x_train, columns=x_train_df.columns).loc[:, rfe.support_]
x_valid_selected = pd.DataFrame(x_valid, columns=x_train_df.columns).loc[:, rfe.support_]

## Hyperparameter Tuning

Now we will tune the Random Forest Regressor hyperparemeters using `GridSearcgCV`.

In [9]:
# Define hyperparameter grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}

# Initialize model
rf_model = RandomForestRegressor(random_state=RANDOM_STATE)

# Grid search with selected features
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

# Fit grid search to the **selected features only**
grid_search.fit(selected_features, y_train)

# Best parameters
print("Best Hyperparameters:", grid_search.best_params_)
print("Best R-squared Score:", grid_search.best_score_)

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Best Hyperparameters: {'bootstrap': False, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best R-squared Score: 0.9528334994553305


Here we see the hyper params for the best model and the R-squared it got. Let's retrieve the best model.

In [12]:
# best model
best_rf_model = grid_search.best_estimator_

## In-Sample Evaluation

In [13]:
# Make predictions on the training data
insample_predictions = best_rf_model.predict(x_train_selected)

# calculate R-squared score for in-sample predictions
print(f"In-Sample Evaluation:")

insample_r2 = r2_score(y_train, insample_predictions)
print(f"  300m buffer zone R-squared: {insample_r2}")

In-Sample Evaluation:
  300m buffer zone R-squared: 1.0


## Out-Sample Evaluation

In [14]:
# Make predictions on the validation data
outsample_predictions = best_rf_model.predict(x_valid_selected)

# calculate R-squared score for in-sample predictions
print(f"Out-Sample Evaluation:")

outsample_r2 = r2_score(y_valid, outsample_predictions)
print(f"  300m buffer zone R-squared: {outsample_r2}")

Out-Sample Evaluation:
  300m buffer zone R-squared: 0.9622428080621714


## Feature Importance

>TODO: apply SHAP here

# Challenge Submission

>TODO

# Summary Table

In [ ]:
# create the summary table
models = ['300m_tuned_RandomForestRegressor_model']

data = {
    'Model': models,
    'Training R-squared': [insample_r2],
    'Validation R-squared': [outsample_r2]
}

summary_df = pd.DataFrame(data)
summary_df

# Save The Model

In [ ]:
# Save the model and scaler to files
with open('300m_tuned_RandomForestRegressor_model.pkl', 'wb') as f:
    pickle.dump(best_rf_model, f)
with open('300m_standard_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Conclusion

>TODO